# 🎵 Estudio de Música Digital — Separar instrumentos con IA

Este notebook separa una canción en pistas individuales (voz, batería, bajo, piano, guitarra, otros) usando **Demucs** (Meta AI), el mejor modelo abierto para esto hoy.

**Por qué en Colab y no en el teléfono/servidor de VAWOL:** este modelo necesita GPU para andar en minutos en vez de correr el riesgo de colgarse por falta de RAM. Colab te da una GPU gratis (sin tarjeta de crédito) — este es el camino de costo cero.

Hay **dos formas** de usarlo (elegí una sección más abajo):

- **A) Manual**: subís una canción, esperás, se descarga un .zip. Simple, un solo uso.
- **B) Servidor vivo**: el notebook queda corriendo y expone una URL pública — desde ahí, **arte.vawol.com** (o el agente Jano) le puede pedir separar canciones directamente, sin bajar/subir nada a mano, mientras esta pestaña de Colab siga abierta.

**Antes que nada**: `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → elegí **GPU (T4)**.

In [ ]:
# Verificar que hay GPU (si dice "no GPU", volvé al paso de arriba)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "⚠️ SIN GPU: anda a Entorno de ejecución > Cambiar tipo > GPU"

In [ ]:
# Instalar Demucs (tarda ~1-2 minutos)
!pip install -q demucs

## A) Modo manual — subir una canción y bajar el .zip

In [ ]:
# Subir la canción
from google.colab import files
import os

subidos = files.upload()
nombre_archivo = list(subidos.keys())[0]
print(f"\n✓ Subido: {nombre_archivo}")

### Elegí el modelo

- **`htdemucs_6s`** (recomendado si querés el **piano** o la **guitarra** separados): 6 pistas — voz, batería, bajo, guitarra, piano, otros. Es un modelo experimental; a veces se "filtra" un poco de otro instrumento en la pista de piano/guitarra.
- **`htdemucs_ft`**: el de mejor calidad general, pero solo 4 pistas — voz, batería, bajo, otros (el piano queda mezclado dentro de "otros"). Tarda un poco más (corre 4 modelos y promedia).

In [ ]:
MODELO = "htdemucs_6s"  # cambiar a "htdemucs_ft" si no necesitás piano/guitarra separados

!demucs -n {MODELO} "{nombre_archivo}"

In [ ]:
# Empaquetar los resultados y descargar
import shutil

base = os.path.splitext(nombre_archivo)[0]
carpeta = f"separated/{MODELO}/{base}"
print("Pistas generadas:", os.listdir(carpeta))

zip_path = f"{base}_instrumentos"
shutil.make_archive(zip_path, "zip", carpeta)
files.download(f"{zip_path}.zip")

---
## B) Modo servidor vivo — para usar directo desde arte.vawol.com

Levanta un mini-servidor (FastAPI) con un endpoint `/separar`, y lo expone a
internet con **cloudflared** (el mismo túnel que ya usamos en el nodo del
teléfono). Mientras esta celda siga corriendo, la URL pública que imprime
puede recibir pedidos de separación reales — el sitio o el MCP server se la
pasan y les devuelve el .zip de instrumentos, sin que nadie tenga que bajar
ni subir nada a mano.

**Importante**: esto NO es un servicio 24/7 — solo funciona mientras tengas
esta pestaña de Colab abierta y conectada. Si se corta la sesión, hay que
volver a correr esta celda (te da una URL nueva cada vez).

In [ ]:
%%writefile servidor.py
# API minima: recibe un audio, corre Demucs, devuelve el .zip de instrumentos.
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
import subprocess, os, shutil, uuid

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

@app.get("/salud")
def salud():
    return {"ok": True}

@app.post("/separar")
async def separar(archivo: UploadFile = File(...), modelo: str = Form("htdemucs_6s")):
    trabajo = str(uuid.uuid4())
    carpeta = f"/tmp/{trabajo}"
    os.makedirs(carpeta, exist_ok=True)
    ruta_entrada = os.path.join(carpeta, archivo.filename)
    with open(ruta_entrada, "wb") as f:
        shutil.copyfileobj(archivo.file, f)

    subprocess.run(["demucs", "-n", modelo, "-o", carpeta, ruta_entrada], check=True)

    base = os.path.splitext(archivo.filename)[0]
    carpeta_salida = os.path.join(carpeta, modelo, base)
    zip_path = shutil.make_archive(os.path.join(carpeta, "instrumentos"), "zip", carpeta_salida)
    return FileResponse(zip_path, media_type="application/zip", filename="instrumentos.zip")

In [ ]:
# Instalar dependencias del servidor + cloudflared, y levantar todo
!pip install -q fastapi uvicorn python-multipart
!test -f cloudflared || (wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared)

import subprocess, time, re

proceso_servidor = subprocess.Popen(
    ["uvicorn", "servidor:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(4)

proceso_tunel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

url_publica = None
for _ in range(40):
    linea = proceso_tunel.stdout.readline()
    m = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", linea)
    if m:
        url_publica = m.group(0)
        break

if url_publica:
    print("✓ Servidor de separación activo en:", url_publica)
    print("  Pegá esa URL en arte.vawol.com (Estudio → Separar con IA → URL del servidor) para esta sesión.")
else:
    print("⚠️ No pude leer la URL del túnel — revisá la salida de arriba.")

In [ ]:
# Correr esta celda para apagar el servidor y el túnel cuando termines
proceso_tunel.terminate()
proceso_servidor.terminate()
print("Servidor y túnel detenidos.")

---
### Notas honestas
- **Tiempo**: con GPU T4, una canción de 3-4 minutos tarda típicamente 1-3 minutos en separarse (vs. varios minutos por CPU, sin GPU).
- **Cuota gratis de Colab**: variable según uso de Google, pero suele rondar 12-30 horas de GPU por semana sin pagar nada. Para separar canciones sueltas alcanza de sobra.
- **Calidad**: Demucs es el estado del arte abierto hoy, pero ningún separador es perfecto — vas a escuchar algo de "bleed" (residuos de otros instrumentos), sobre todo en el modelo de 6 pistas.
- **El modo servidor no es 24/7**: necesita que tengas la pestaña de Colab abierta y conectada. Si se corta la sesión (por inactividad o límite de tiempo), hay que volver a correr las celdas — te va a dar una URL nueva cada vez.